<a href="https://colab.research.google.com/github/UbaidMalik365/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UbaidMalik365/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os

os.chdir("/content/flyrank-ml-internship-starter")

print("Current folder:", os.getcwd())
print("CSV exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Current folder: /content/flyrank-ml-internship-starter
CSV exists: True


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Finding 1: Search visibility and content performance

The research paper reports findings about factors associated with search performance and visibility.

**Methodology question:** How was the outcome or label defined, and what time window was used to create it? I would want to know whether the label represents a clearly measured future outcome rather than a value that was already available when the prediction features were collected.

**Validation question:** Does the validation design separate the data used to develop the model from the data used to evaluate the claim? In particular, I would check whether pages or observations from the same entities could appear in both development and evaluation data.

### Finding 2: Relationships between content/search signals and outcomes

The research paper reports relationships between observable content or search signals and performance outcomes.

**Methodology question:** Are these relationships observational or supported by an experimental design? I would want to distinguish an observed association from evidence that changing a particular signal would cause better search performance.

**Validation question:** Does the validation design support the scope of the reported claim? I would check whether the evaluation covers data that is separate in time or by group from the data used to develop the analysis, and whether the results remain consistent under that validation approach.

These questions are intended as constructive methodology checks. They help distinguish what the evidence measures from what the evidence can reasonably claim.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

# Create target
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

# Features used in Week 5
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].fillna(0)
y = df["is_declining_label"]

# -----------------------------
# BEFORE: Random split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

random_model.fit(X_train, y_train)

random_scores = random_model.predict_proba(X_test)[:, 1]

random_top50 = (
    pd.DataFrame({
        "actual": y_test.values,
        "score": random_scores
    })
    .sort_values("score", ascending=False)
    .head(50)
)

random_precision_50 = random_top50["actual"].mean()


# -----------------------------
# AFTER: Grouped split by client
# -----------------------------
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

group_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

group_model.fit(X_train_group, y_train_group)

group_scores = group_model.predict_proba(
    X_test_group
)[:, 1]

group_top50 = (
    pd.DataFrame({
        "actual": y_test_group.values,
        "score": group_scores
    })
    .sort_values("score", ascending=False)
    .head(50)
)

group_precision_50 = group_top50["actual"].mean()


# -----------------------------
# Comparison
# -----------------------------
comparison = pd.DataFrame({
    "Validation design": [
        "Week-5 random split",
        "Week-6 grouped by client"
    ],
    "Precision@50": [
        random_precision_50,
        group_precision_50
    ]
})

display(comparison)

print("\nRandom split Precision@50:", round(random_precision_50, 2))
print("Grouped-by-client Precision@50:", round(group_precision_50, 2))

print("\nTraining clients:",
      df.iloc[train_idx]["client_id"].nunique())

print("Testing clients:",
      df.iloc[test_idx]["client_id"].nunique())

print("Shared clients between train and test:",
      len(
          set(df.iloc[train_idx]["client_id"])
          &
          set(df.iloc[test_idx]["client_id"])
      ))

,Validation design,Precision@50
0,Week-5 random split,0.70
1,Week-6 grouped by client,0.58



Random split Precision@50: 0.7
Grouped-by-client Precision@50: 0.58

Training clients: 25
Testing clients: 7
Shared clients between train and test: 0


## 2. My model under an honest split (before/after)

The Week-5 random split produced a Precision@50 of 0.70. When I re-ran the same Decision Tree using a grouped-by-client split, Precision@50 decreased to 0.58.

The grouped split is more conservative because pages from the same client are not shared between training and testing data. The training set contained 25 clients and the testing set contained 7 clients, with 0 shared clients.

This result suggests that the random-split performance may have been optimistic. The grouped result is a more realistic estimate of how the model may perform on clients it has not seen during training.

The comparison is directional and supports decision-making; it does not prove that the model will improve content performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [13]:
# Final features used by the model
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Columns that could directly reveal the target
leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Final model features:")
for feature in features:
    print(" -", feature)

print("\nLeakage candidates checked:")
for feature in leakage_candidates:
    print(" -", feature)

# Check that no leakage candidate is included in the feature list
leaked_features = set(features).intersection(leakage_candidates)

print("\nFeatures accidentally included from leakage candidates:",
      leaked_features)

if len(leaked_features) == 0:
    print("LEAKAGE CHECK: PASS")
else:
    print("LEAKAGE CHECK: REVIEW REQUIRED")

Final model features:
 - content_age_days
 - days_since_last_update
 - impressions_90d
 - avg_position
 - ctr
 - word_count

Leakage candidates checked:
 - trend_direction
 - trend_pct
 - is_declining_label

Features accidentally included from leakage candidates: set()
LEAKAGE CHECK: PASS


## 3. Leakage audit

I checked the final feature set for target leakage and future information.

The model uses `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, and `word_count`.

I deliberately exclude `trend_direction` and `trend_pct` because they are directly related to the target and could reveal whether a page is declining.

I also exclude `is_declining_label` itself because it is the target being predicted, not a feature.

The remaining features represent observable page and performance information available for the decision. This audit is intended to reduce leakage risk, although it cannot prove that every possible source of temporal leakage is absent.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

My original claim was:

"The Decision Tree is better than the baseline and can identify declining pages."

A safer claim is:

"On the measured test data, the Decision Tree achieved higher Precision@50 than the Week-4 baseline under the random split (0.62 vs 0.46). Under the grouped-by-client split, Precision@50 was 0.58. These results provide directional evidence that the Decision Tree can support prioritization of potentially declining pages, but they do not prove that the model will improve page performance or generalize equally well to unseen clients."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.